In [1]:
from core.model_providers.openai_llm import OpenAILLM
from core import KAGConfig
from core.storage.graph_store import GraphStore
from core.utils.neo4j_utils import Neo4jUtils

config = KAGConfig.from_yaml("configs/config_openai.yaml")
llm = OpenAILLM(config)
doc_type = config.knowledge_graph_builder.doc_type
graph_store = GraphStore(config)
neo4j_utils = Neo4jUtils(graph_store, doc_type=doc_type)

/vepfs-mlp2/c20250513/241404044/users/roytian/anaconda3/envs/screenplay/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
episode_list = neo4j_utils.fetch_all_nodes(node_types=["Episode"])

In [3]:
episode_list[0]

{'labels': ['Entity', 'Episode'],
 'id': 'ep_208aeb02d095e737',
 'name': 'Carl heats a Lean Cuisine',
 'description': 'Carl places a Lean Cuisine in the microwave and turns it on to heat the meal.',
 'properties': '{"related_events": ["Carl heats a Lean Cuisine in the microwave (scene_10)"]}',
 'source_documents': ['scene_10_part_1']}

In [4]:
def find_episode_chain(utils, entity_id: str, min_confidence: float = 0.0):
    # 关系类型集合（含旧名，确保兼容）
    rel_types = [
        "EPISODE_CAUSAL_LINK"
    ]
    rel_types_str = "|".join(rel_types)

    cypher = f"""
    MATCH path = (start:Episode {{id: $entity_id}})-[
        r:{rel_types_str}*
    0..]->(end:Episode)
    WHERE
    // 路径上所有关系满足置信度阈值
    ALL(rel IN relationships(path)
        WHERE coalesce(rel.confidence, 0.0) >= $min_confidence)
    // 终点：在所考虑的关系集合中不再有出边
    AND NOT (end)-[:{rel_types_str}]->()
    RETURN [n IN nodes(path) | n.id] AS episode_chain
    """

    results = utils.execute_query(
        cypher,
        {
            "entity_id": entity_id,
            "min_confidence": float(min_confidence)
        }
    )
    return [record["episode_chain"] for record in results if "episode_chain" in record]

In [5]:
find_episode_chain(neo4j_utils, entity_id="ep_7a2bd6c48226d366")

[]

In [6]:
def get_starting_episodes(utils):
    cypher = """
    MATCH (e:Episode)
    WHERE NOT ()-[:EPISODE_CAUSAL_LINK]->(e)
    RETURN e.id AS episode_id
    """

    result = utils.execute_query(cypher)
    result = [e["episode_id"] for e in result]
    return result

In [7]:
starting_episodes = get_starting_episodes(neo4j_utils)

In [8]:
len(starting_episodes)

73

In [9]:
def get_all_episode_chains(utils, starting_episodes, min_length: int = 3):
    all_chains = []
    for episode_id in starting_episodes:
        chains = find_episode_chain(utils, entity_id=episode_id)
        if len(chains) >= min_length:
            all_chains.extend(chains)
    return all_chains

In [10]:
all_chains = get_all_episode_chains(neo4j_utils, starting_episodes, min_length=2)

In [11]:
len(all_chains)

437

In [12]:
from core.builder.event_processor import (
    segment_trunks_and_branches,
    remove_subset_paths,
    remove_similar_paths,
)

In [13]:
all_segments = segment_trunks_and_branches(all_chains, include_cutpoint=True, min_len=3)

In [14]:
len(all_segments)

25

In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

from typing import Any, Dict, Iterable, List, Optional, Set, Tuple

import networkx as nx


def build_episode_index(episodes: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """id -> episode_obj"""
    idx: Dict[str, Dict[str, Any]] = {}
    for ep in episodes:
        eid = str(ep.get("id", "")).strip()
        if eid:
            idx[eid] = ep
    return idx


def iter_edges_from_relations(
    relations: List[Dict[str, Any]],
    *,
    predicate: str = "EPISODE_CAUSAL_LINK",
) -> Iterable[Tuple[str, str]]:
    """Yield (u, v) only from relations whose predicate matches."""
    for r in relations:
        if str(r.get("predicate", "")).strip() != predicate:
            continue
        u = str(r.get("subject_id", "")).strip()
        v = str(r.get("object_id", "")).strip()
        if u and v:
            yield (u, v)


def build_rel_by_uv(
    relations: List[Dict[str, Any]],
    *,
    predicate: str = "EPISODE_CAUSAL_LINK",
) -> Dict[Tuple[str, str], Dict[str, Any]]:
    """(u,v) -> relation_obj for the given predicate."""
    m: Dict[Tuple[str, str], Dict[str, Any]] = {}
    for r in relations:
        if str(r.get("predicate", "")).strip() != predicate:
            continue
        u = str(r.get("subject_id", "")).strip()
        v = str(r.get("object_id", "")).strip()
        if u and v:
            m[(u, v)] = r
    return m


def make_edge_filter(
    rel_by_uv: Dict[Tuple[str, str], Dict[str, Any]],
    *,
    min_effective_weight: float = 0.0,
    min_similarity: Optional[float] = None,
    min_common_neighbors: Optional[int] = None,
) -> Any:
    """
    Edge filter using ONLY EPISODE_CAUSAL_LINK relations (already ensured by rel_by_uv).

    Uses your fields:
      - effective_weight
      - properties.similarity
      - properties.common_neighbors
    """
    def _f(u: str, v: str) -> bool:
        r = rel_by_uv.get((u, v))
        if not r:
            return False

        # effective_weight
        ew = r.get("effective_weight", 0.0)
        try:
            ew = float(ew) if ew is not None else 0.0
        except Exception:
            ew = 0.0
        if ew < float(min_effective_weight):
            return False

        props = r.get("properties", {}) or {}

        if min_similarity is not None:
            try:
                sim = float(props.get("similarity", 0.0) or 0.0)
            except Exception:
                sim = 0.0
            if sim < float(min_similarity):
                return False

        if min_common_neighbors is not None:
            try:
                cn = int(props.get("common_neighbors", 0) or 0)
            except Exception:
                cn = 0
            if cn < int(min_common_neighbors):
                return False

        return True

    return _f


def dag_minimum_path_cover(
    nodes: Iterable[str],
    edges: Iterable[Tuple[str, str]],
    *,
    edge_filter_fn: Optional[Any] = None,
    check_is_dag: bool = True,
) -> List[List[str]]:
    """
    Minimum path cover for a DAG using maximum bipartite matching.

    Returns:
      paths: list of paths (each is a list of node ids). Each node appears exactly once.
    """
    V: List[str] = list(nodes)
    V_set: Set[str] = set(V)

    used_edges: List[Tuple[str, str]] = []
    for u, v in edges:
        if u in V_set and v in V_set:
            if edge_filter_fn is None or bool(edge_filter_fn(u, v)):
                used_edges.append((u, v))

    if check_is_dag:
        G = nx.DiGraph()
        G.add_nodes_from(V)
        G.add_edges_from(used_edges)
        if not nx.is_directed_acyclic_graph(G):
            raise ValueError("Input graph is not a DAG after filtering. Minimum path cover requires a DAG.")

    # Build bipartite graph
    B = nx.Graph()
    left_nodes = [("L", u) for u in V]
    right_nodes = [("R", u) for u in V]
    B.add_nodes_from(left_nodes, bipartite=0)
    B.add_nodes_from(right_nodes, bipartite=1)

    for u, v in used_edges:
        B.add_edge(("L", u), ("R", v))

    matching = nx.algorithms.bipartite.matching.hopcroft_karp_matching(
        B, top_nodes=set(left_nodes)
    )

    successor: Dict[str, str] = {}
    predecessor: Dict[str, str] = {}

    for lu in left_nodes:
        if lu in matching:
            rv = matching[lu]
            if isinstance(rv, tuple) and len(rv) == 2 and rv[0] == "R":
                u = lu[1]
                v = rv[1]
                successor[u] = v
                predecessor[v] = u

    starts = [u for u in V if u not in predecessor]

    paths: List[List[str]] = []
    visited: Set[str] = set()

    for s in starts:
        cur = s
        path = [cur]
        visited.add(cur)
        while cur in successor:
            nxt = successor[cur]
            if nxt in visited:
                break
            path.append(nxt)
            visited.add(nxt)
            cur = nxt
        paths.append(path)

    for u in V:
        if u not in visited:
            paths.append([u])
            visited.add(u)

    return paths


def extract_storyline_trunks_by_mpc(
    episodes: List[Dict[str, Any]],
    relations: List[Dict[str, Any]],
    *,
    predicate: str = "EPISODE_CAUSAL_LINK",
    min_effective_weight: float = 0.0,
    min_similarity: Optional[float] = None,
    min_common_neighbors: Optional[int] = None,
    check_is_dag: bool = True,
    attach_debug_text: bool = False,
) -> List[Dict[str, Any]]:
    """
    MPC trunks using ONLY relations with predicate == "EPISODE_CAUSAL_LINK".

    Returns:
      [
        {"trunk": [...], "trunk_data": [...]? },
        ...
      ]
    """
    ep_idx = build_episode_index(episodes)
    nodes = list(ep_idx.keys())

    rel_by_uv = build_rel_by_uv(relations, predicate=predicate)
    edge_filter_fn = make_edge_filter(
        rel_by_uv,
        min_effective_weight=min_effective_weight,
        min_similarity=min_similarity,
        min_common_neighbors=min_common_neighbors,
    )

    edges = list(iter_edges_from_relations(relations, predicate=predicate))

    paths = dag_minimum_path_cover(
        nodes,
        edges,
        edge_filter_fn=edge_filter_fn,
        check_is_dag=check_is_dag,
    )

    # Sort: longer trunks first
    paths = sorted(paths, key=lambda p: (-len(p), p))

    out: List[Dict[str, Any]] = []
    for p in paths:
        item: Dict[str, Any] = {"trunk": p}
        if attach_debug_text:
            item["trunk_data"] = [
                {
                    "id": eid,
                    "name": ep_idx.get(eid, {}).get("name", ""),
                    "description": ep_idx.get(eid, {}).get("description", ""),
                    "source_documents": ep_idx.get(eid, {}).get("source_documents", []),
                }
                for eid in p
            ]
        out.append(item)

    return out



In [16]:
from core.utils.general_utils import load_json

episodes = load_json("data/narrative_graph/global/episodes.json")
relations = load_json("data/narrative_graph/global/episode_relations_dag.json")

# Example parameters:
# - your candidates were already constrained by similarity>=0.6 and common_neighbors>=1,
#   so you can set the same thresholds again here to be safe.
storylines = extract_storyline_trunks_by_mpc(
        episodes=episodes,
        relations=relations,
        predicate="EPISODE_CAUSAL_LINK",
        min_effective_weight=0.4,
        min_similarity=0.6,
        min_common_neighbors=1,
        check_is_dag=True,
        attach_debug_text=True,
    )
print("num trunks:", len(storylines))
print("top trunk:", storylines[0] if storylines else None)

num trunks: 81
top trunk: {'trunk': ['ep_fd33b654339313e3', 'ep_6dc13158d6c6c000', 'ep_9caf77073a7d7b1f', 'ep_8fc74e1b470cef21', 'ep_2811a77de56e516e', 'ep_73a82759dc35089e', 'ep_e4a9dd27bf3ea8a0', 'ep_8647579437e8fb34', 'ep_93c74e669a717eb1', 'ep_a33d11557de45b60'], 'trunk_debug': [{'id': 'ep_fd33b654339313e3', 'name': 'Norman invites Carl to a funny hat party', 'description': 'Norman tells Carl about a party he is hosting this Friday, describing it as a funny hat party. He personally invites Carl and asks him to attend. Carl checks his calendar, sees it is empty, and responds by saying he will be out of town.', 'source_documents': ['scene_7_part_1']}, {'id': 'ep_6dc13158d6c6c000', 'name': 'Norman reminds Carl of the party as the hour changes', 'description': 'The clock clicks forward to mark the beginning of a new hour. Norman speaks to Carl, reminding him about the upcoming party. Carl looks up at the clock and responds, saying he will think about the party.', 'source_documents': ['

In [ ]:
def filter_storylines_by_min_length_with_stats(
    storylines: List[Dict[str, Any]],
    *,
    min_length: int = 3,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """
    Filter storylines by trunk length >= min_length.

    Returns:
      filtered_storylines,
      stats = {
        "before": int,
        "after": int,
        "removed": int,
        "min_length": int,
        "length_histogram": {length: count}
      }
    """
    from collections import Counter

    lengths = [len(sl.get("trunk", [])) for sl in storylines]
    hist = dict(Counter(lengths))

    filtered = [
        sl for sl in storylines
        if len(sl.get("trunk", [])) >= min_length
    ]

    stats = {
        "before": len(storylines),
        "after": len(filtered),
        "removed": len(storylines) - len(filtered),
        "min_length": min_length,
        "length_histogram": hist,
    }

    return filtered, stats 


In [18]:
storylines, stats = filter_storylines_by_min_length_with_stats(
    storylines,
    min_length=3,
)

print(stats)


{'before': 81, 'after': 13, 'removed': 68, 'min_length': 3, 'length_histogram': {10: 1, 6: 1, 5: 2, 4: 2, 3: 7, 2: 12, 1: 56}}


In [19]:
len(storylines)

13

In [20]:
len(storylines[0]["trunk"])

10

In [28]:
content = "\n".join([f"{chunk["name"]}: {chunk["description"]}" for chunk in  storylines[0]["trunk_debug"]])
print(content)

Norman invites Carl to a funny hat party: Norman tells Carl about a party he is hosting this Friday, describing it as a funny hat party. He personally invites Carl and asks him to attend. Carl checks his calendar, sees it is empty, and responds by saying he will be out of town.
Norman reminds Carl of the party as the hour changes: The clock clicks forward to mark the beginning of a new hour. Norman speaks to Carl, reminding him about the upcoming party. Carl looks up at the clock and responds, saying he will think about the party.
Carl returns to loan desk after promotion talk: After the conversation with Norman about the job offer and the upcoming party, Carl walks back to the loan desk, visibly pleased about the prospect of his new position.
Carl arrives and is greeted by the group: Carl enters the bar where Peter, Lucy, and Rooney are already present. Rooney reacts with surprise, exclaiming that he can't believe Carl is out in the world and jokingly asking if he's having a religious

In [27]:
storylines[0]["trunk_debug"]

[{'id': 'ep_fd33b654339313e3',
  'name': 'Norman invites Carl to a funny hat party',
  'description': 'Norman tells Carl about a party he is hosting this Friday, describing it as a funny hat party. He personally invites Carl and asks him to attend. Carl checks his calendar, sees it is empty, and responds by saying he will be out of town.',
  'source_documents': ['scene_7_part_1']},
 {'id': 'ep_6dc13158d6c6c000',
  'name': 'Norman reminds Carl of the party as the hour changes',
  'description': 'The clock clicks forward to mark the beginning of a new hour. Norman speaks to Carl, reminding him about the upcoming party. Carl looks up at the clock and responds, saying he will think about the party.',
  'source_documents': ['scene_9_part_1']},
 {'id': 'ep_9caf77073a7d7b1f',
  'name': 'Carl returns to loan desk after promotion talk',
  'description': 'After the conversation with Norman about the job offer and the upcoming party, Carl walks back to the loan desk, visibly pleased about the pro

In [2]:
from qwen_agent.tools.base import BaseTool, register_tool

class DocMappingLookupTool(BaseTool):
    """
    Query title/subtitle <-> source_documents mappings generated by prepare_chunks.
    """

    name = "doc_mapping_lookup"
    description = (
        "查询 title/subtitle 与 source_documents 的双向映射。"
        "其中 source_documents 就是 document-level vector store 的 ID（例如 document_144_part_1）。"
        "常用于图节点证据回链：先拿节点 source_documents，再反查标题，或反向由标题查 source_documents。"
        "支持 title_to_source_documents 和 source_documents_to_title，并支持模糊匹配。"
    )
    parameters = [
        {
            "name": "mapping_type",
            "type": "string",
            "description": (
                "映射类型：title_to_source_documents | source_documents_to_title。"
                "兼容旧别名：title_to_document_ids / document_id_to_title"
            ),
            "required": True,
        },
        {
            "name": "key",
            "type": "string",
            "description": "查询键（title/subtitle 组合键，或 source_documents）",
            "required": True,
        },
        {"name": "fuzzy", "type": "boolean", "description": "是否开启模糊匹配，默认 true", "required": False},
        {"name": "limit", "type": "integer", "description": "最多返回条数，默认 10", "required": False},
    ]


In [ ]:
tool = DocMappingLookupTool

TypeError: Can't instantiate abstract class DocMappingLookupTool without an implementation for abstract method 'call'

In [7]:
tool.description

'查询 title/subtitle 与 source_documents 的双向映射。其中 source_documents 就是 document-level vector store 的 ID（例如 document_144_part_1）。常用于图节点证据回链：先拿节点 source_documents，再反查标题，或反向由标题查 source_documents。支持 title_to_source_documents 和 source_documents_to_title，并支持模糊匹配。'